# 📡 Telecom Customer Churn Analysis
## Exploratory Data Analysis | Python

---

## 📋 Project Summary

This notebook performs a comprehensive **Exploratory Data Analysis (EDA)** on a Telecom Customer Churn dataset to uncover patterns and drivers behind customer attrition.

### Why This Matters
Telecom companies lose significant revenue to churn every year. Understanding *why* customers leave enables:
- Targeted retention campaigns for at-risk customers
- Better contract and pricing strategies
- Improved service offerings to reduce dissatisfaction
- Data-driven foundation for predictive churn modeling

---

## 📊 Dataset Overview

| Detail | Info |
|---|---|
| **Source** | Kaggle — Telco Customer Churn |
| **File** | `Customer Churn.csv` |
| **Records** | ~7,043 customers |
| **Features** | 21 columns (demographics, services, billing, churn) |
| **Target** | `Churn` — Yes / No |

---

## 🔍 Analysis Sections

| Section | Description |
|---|---|
| **1** | Import Libraries |
| **2** | Load Data |
| **3** | Data Audit & Cleaning |
| **4** | Churn Distribution |
| **5** | Demographic Analysis (Gender, Senior Citizen) |
| **6** | Behavioral Analysis (Tenure, Contract) |
| **7** | Service Usage Analysis |
| **8** | Payment Method Analysis |

## Section 1: Import Libraries

**Goal:** Load all Python libraries required for data manipulation and visualisation.

| Library | Purpose |
|---|---|
| `pandas` | DataFrame operations, groupby, aggregation |
| `numpy` | Numerical operations |
| `matplotlib` | Base plotting framework |
| `seaborn` | Statistical visualisations built on matplotlib |
| `os` | File path and directory management |

In [ ]:
# ============================================================
# SECTION 1: Import Libraries
# ============================================================
# PURPOSE: Load all Python libraries needed for this project.
#
# Library groups:
#   - Data manipulation : pandas, numpy
#   - Visualisation     : matplotlib, seaborn
#   - Utilities         : os (directory management)
#
# Why these specific libraries?
#   - seaborn is built on matplotlib and provides higher-level statistical
#     plots (countplot, histplot) that work directly with DataFrames.
#   - pandas groupby + value_counts enable quick aggregate statistics
#     without writing manual loops.
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress non-critical warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

## Section 2: Load Data

**Goal:** Read the Telecom Customer Churn CSV into a pandas DataFrame and perform an initial inspection.

> **Note:** Update the file path below to match your local environment before running.

In [ ]:
# ============================================================
# SECTION 2: Load Data
# ============================================================
# PURPOSE: Read the Customer Churn CSV file into a pandas DataFrame.
#
# DESIGN DECISION:
#   - pd.read_csv() is used as the source file is a flat CSV.
#   - We immediately display shape and head() to confirm successful load
#     and get a first look at data values and formatting.
#
# NOTE: Update the file path below to match your local environment.
# ============================================================

# ── 2.1 Set working directory ────────────────────────────────
# os.chdir('path/to/your/data/folder')   # ← update this path if needed

# ── 2.2 Load dataset ────────────────────────────────────────
df = pd.read_csv('Customer Churn.csv')

print(f"✅ Dataset loaded successfully!")
print(f"   Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print()
df.head(5)

## Section 3: Data Audit & Cleaning

**Goal:** Understand the dataset's structure, identify quality issues, and apply targeted cleaning steps before any visualisation.

Key questions answered here:
1. What are the column names and data types?
2. Are there missing or null values?
3. Are there duplicate records?
4. Do any columns need type conversion or re-encoding?

**Why audit before analysis?** Dirty data leads to misleading charts and incorrect conclusions. Catching issues early avoids model-breaking surprises downstream.

In [ ]:
# ============================================================
# SECTION 3.1: Data Audit — Shape, Dtypes & Columns
# ============================================================
# PURPOSE: Get a high-level structural overview of the dataset.
#   - info()    : column names, non-null counts, dtypes in one view
#   - columns   : confirm exact column names (useful for later references)
#
# TotalCharges is expected to be numeric but may be read as object
# due to blank strings for new customers — flagged here and fixed in 3.3.
# ============================================================

print("=" * 60)
print("DATASET AUDIT")
print("=" * 60)

print("\nColumn info:")
df.info()

print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# ============================================================
# SECTION 3.2: Missing Value Check
# ============================================================
# PURPOSE: Identify columns with null values and quantify the extent.
#
# INTERPRETATION:
#   - A non-zero count on TotalCharges indicates blank strings
#     (not true NaN), which isnull() will NOT catch here.
#   - Those blank strings are handled separately in Section 3.3.
# ============================================================

print("=" * 60)
print("MISSING VALUE REPORT")
print("=" * 60)

missing = df.isnull().sum()
if (missing > 0).any():
    print(missing[missing > 0])
else:
    print("✅ No null values found.")

print(f"\nDuplicate rows    : {df.duplicated().sum()}")
print(f"Duplicate Customer IDs : {df['customerID'].duplicated().sum()}")

In [ ]:
# ============================================================
# SECTION 3.3: Fix TotalCharges — Blank Strings → Float
# ============================================================
# PURPOSE: TotalCharges is stored as object dtype because new customers
# with Tenure = 0 have blank strings instead of 0.
#
# FIX STRATEGY:
#   1. Replace blank strings " " with "0" (new customers have no charges yet)
#   2. Cast the column to float64 for numerical operations
#
# WHY 0 AND NOT NaN?
#   These are valid customers with zero charges — dropping or marking them
#   as NaN would remove real data points from the analysis.
# ============================================================

df['TotalCharges'] = df['TotalCharges'].replace(" ", "0")
df['TotalCharges'] = df['TotalCharges'].astype("float")

print("✅ TotalCharges converted to float.")
print(f"   dtype is now: {df['TotalCharges'].dtype}")

In [ ]:
# ============================================================
# SECTION 3.4: Encode SeniorCitizen — 0/1 → 'No'/'Yes'
# ============================================================
# PURPOSE: SeniorCitizen is stored as a binary integer (0 or 1).
#   Converting it to 'No'/'Yes' makes plots self-explanatory without
#   needing a separate legend note, and keeps it consistent with other
#   binary Yes/No columns in this dataset.
# ============================================================

df['SeniorCitizen'] = df['SeniorCitizen'].map({1: "Yes", 0: "No"})

print("✅ SeniorCitizen encoded: 0 → 'No', 1 → 'Yes'")
print()
df.head(3)

In [ ]:
# ============================================================
# SECTION 3.5: Statistical Summary
# ============================================================
# PURPOSE: Review the statistical range of all numeric columns.
#   - Spot obvious outliers (e.g., negative values, implausible maxima)
#   - Understand the scale of MonthlyCharges, TotalCharges, Tenure
#   - Confirm that the cleaning steps above produced valid numeric ranges
# ============================================================

print("Statistical Summary:")
df.describe()

## Section 4: Churn Distribution

**Goal:** Understand the overall scale of churn in this dataset before drilling into individual factors.

Key questions:
- How many customers have churned vs. stayed?
- What percentage of the customer base does churn represent?

**Why start here?** This establishes the baseline churn rate that all subsequent segment analyses will be compared against.

In [ ]:
# ============================================================
# SECTION 4.1: Churn Count — Bar Chart
# ============================================================
# PURPOSE: Display the raw count of churned vs. retained customers.
#
# bar_label() adds exact counts on top of each bar, making the
# magnitude immediately readable without needing to estimate from
# the y-axis scale.
# ============================================================

plt.figure(figsize=(4, 4))
ax = sns.countplot(x='Churn', data=df, palette=['#1f77b4', '#ff7f0e'])
ax.bar_label(ax.containers[0], fontsize=11)
plt.title('Count of Customers by Churn', fontsize=13)
plt.xlabel('Churn')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECTION 4.2: Churn Distribution — Pie Chart
# ============================================================
# PURPOSE: Show churn as a percentage of the total customer base.
#
# A pie chart is effective here because we are communicating a simple
# two-category proportion — the "26.54% have churned" headline figure
# is the key business message.
# ============================================================

gb = df.groupby("Churn").agg({'Churn': 'count'})

plt.figure(figsize=(4, 4))
plt.pie(
    gb['Churn'],
    labels=gb.index,
    autopct="%1.2f%%",
    colors=['#1f77b4', '#ff7f0e'],
    startangle=90
)
plt.title("Churn Distribution (%)", fontsize=12)
plt.tight_layout()
plt.show()

> **📌 Insight:** ~26.54% of customers in this dataset have churned. This is the baseline churn rate we aim to understand and ultimately reduce.

## Section 5: Demographic Analysis

**Goal:** Examine how customer demographics — gender and senior citizen status — relate to churn.

### Why demographics matter:
- Demographic segments respond differently to retention strategies.
- Identifying over-represented churn groups enables targeted outreach.
- Gender and age are quick filters for personalised marketing.

In [ ]:
# ============================================================
# SECTION 5.1: Churn by Gender
# ============================================================
# PURPOSE: Check whether gender is a meaningful predictor of churn.
#
# hue='Churn' splits each gender bar into churned/retained stacks,
# enabling direct visual comparison of churn rates across genders.
#
# EXPECTED FINDING: Churn rates are roughly equal across male and female
# customers — gender alone is not a strong differentiator.
# ============================================================

plt.figure(figsize=(5, 4))
ax = sns.countplot(x="gender", data=df, hue="Churn", palette=['#1f77b4', '#ff7f0e'])
for container in ax.containers:
    ax.bar_label(container, fontsize=9)
plt.title("Churn by Gender", fontsize=13)
plt.xlabel("Gender")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

> **📌 Insight:** Churn rates are nearly identical across male and female customers. Gender is **not** a significant predictor of churn in this dataset.

In [ ]:
# ============================================================
# SECTION 5.2: Senior Citizen — Customer Count
# ============================================================
# PURPOSE: Understand how many customers are senior citizens vs. not.
#
# This is a prerequisite before comparing churn rates — if senior
# citizens are a very small group, percentage-based comparisons carry
# more uncertainty and need to be interpreted with care.
# ============================================================

plt.figure(figsize=(4, 4))
ax = sns.countplot(x="SeniorCitizen", data=df, palette=['#2ca02c', '#d62728'])
ax.bar_label(ax.containers[0], fontsize=11)
plt.title("Customer Count by Senior Citizen Status", fontsize=12)
plt.xlabel("Senior Citizen")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECTION 5.3: Churn Rate by Senior Citizen Status (Stacked Bar)
# ============================================================
# PURPOSE: Compare churn *rates* (percentages) between senior and
# non-senior customers using a stacked 100% bar chart.
#
# WHY STACKED PERCENTAGE CHART (not raw counts)?
#   - Senior citizens are a much smaller group than non-seniors.
#   - Raw counts would be misleading — percentage normalisation lets us
#     compare churn propensity fairly across unequal group sizes.
#
# INTERPRETATION:
#   - A taller 'Yes' stack for SeniorCitizen='Yes' means proportionally
#     more seniors churn than non-seniors.
# ============================================================

total_counts = (
    df.groupby('SeniorCitizen')['Churn']
    .value_counts(normalize=True)
    .unstack() * 100
)

fig, ax = plt.subplots(figsize=(5, 4))
total_counts.plot(kind='bar', stacked=True, ax=ax, color=['#1f77b4', '#ff7f0e'])

# Add percentage labels inside each bar segment
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 2:   # skip tiny slivers
        ax.text(x + width / 2, y + height / 2, f'{height:.1f}%',
                ha='center', va='center', fontsize=10, color='white', fontweight='bold')

plt.title('Churn Rate by Senior Citizen Status', fontsize=13)
plt.xlabel('Senior Citizen')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(title='Churn', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

> **📌 Insight:** A significantly higher **percentage** of senior citizens churn compared to non-seniors. This makes senior citizens a high-priority retention segment.

## Section 6: Behavioral Analysis — Tenure & Contract

**Goal:** Understand how customer longevity and contract type affect churn likelihood.

### Why these factors matter:
- **Tenure** reveals the critical early window when customers are most at risk.
- **Contract type** is one of the strongest levers a business can pull — locking customers into annual plans dramatically reduces churn.

In [ ]:
# ============================================================
# SECTION 6.1: Churn by Tenure — Histogram
# ============================================================
# PURPOSE: Visualise how long customers stay before churning.
#
# WHY A HISTOGRAM (not a box plot)?
#   - Tenure is a continuous variable ranging from 0–72 months.
#   - A histogram with bins=72 gives one bar per month, revealing
#     the precise timing pattern of churn events.
#   - hue='Churn' overlays churned vs. retained distributions in the
#     same plot for direct comparison.
#
# EXPECTED FINDING:
#   - High churn concentration in months 1–6 (early attrition).
#   - Long-tenured customers (>24 months) rarely churn — loyalty builds over time.
# ============================================================

plt.figure(figsize=(10, 4))
sns.histplot(x="tenure", data=df, bins=72, hue="Churn",
             palette=['#1f77b4', '#ff7f0e'])
plt.title("Churn by Tenure (Monthly Distribution)", fontsize=13)
plt.xlabel("Tenure (Months)")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

> **📌 Insight:** Customers who churn predominantly do so within the **first 1–6 months**. Long-tenured customers (2+ years) are highly stable. Early onboarding experience is critical.

In [ ]:
# ============================================================
# SECTION 6.2: Churn by Contract Type
# ============================================================
# PURPOSE: Compare churn rates across the three contract types:
#   - Month-to-month  (low commitment, highest churn risk)
#   - One year        (medium commitment)
#   - Two year        (high commitment, lowest churn risk)
#
# bar_label() on containers[0] and containers[1] labels both the
# 'No' and 'Yes' churn bars within each contract group.
#
# BUSINESS IMPLICATION:
#   Month-to-month customers represent the easiest quick-win for
#   retention teams — converting them to annual contracts directly
#   reduces churn probability.
# ============================================================

plt.figure(figsize=(6, 4))
ax = sns.countplot(x="Contract", data=df, hue="Churn", palette=['#1f77b4', '#ff7f0e'])
for container in ax.containers:
    ax.bar_label(container, fontsize=9)
plt.title("Churn by Contract Type", fontsize=13)
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

> **📌 Insight:** Month-to-month customers churn at a dramatically higher rate. Customers on 1- or 2-year contracts are far more loyal. **Contract type is the single strongest predictor of churn** in this dataset.

## Section 7: Service Usage Analysis

**Goal:** Understand how subscription to individual services correlates with churn.

The 9 services analysed are:
`PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`

### Why service usage matters:
- Customers with more services are more embedded in the ecosystem → harder to leave.
- Protective services (Security, Backup, Support) may directly address pain points that cause churn.

In [ ]:
# ============================================================
# SECTION 7: Service Usage vs. Churn — Subplot Grid
# ============================================================
# PURPOSE: Plot churn distribution for each of 9 service features in
# a single organised 3×3 subplot grid.
#
# DESIGN DECISIONS:
#   - 3×3 grid keeps all service plots visible without scrolling.
#   - hue='Churn' on each countplot reveals churn split for every
#     service category (Yes/No/No internet service).
#   - tight_layout() prevents label overlap across subplots.
#   - Extra empty subplots are removed cleanly with fig.delaxes().
#
# READING THE CHARTS:
#   - For each service, look at whether customers with "No" service
#     have a proportionally taller orange (Churn=Yes) bar.
#   - If yes → absence of that service is a churn risk factor.
# ============================================================

service_columns = [
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

n_cols = 3
n_rows = (len(service_columns) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(service_columns):
    sns.countplot(x=col, data=df, hue="Churn", ax=axes[i],
                  palette=['#1f77b4', '#ff7f0e'])
    axes[i].set_title(f'Churn by {col}', fontsize=11)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=15)

# Remove any unused subplot axes
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Service Usage vs. Churn", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

> **📌 Insight:** Customers **without** `OnlineSecurity`, `TechSupport`, and `OnlineBackup` show noticeably higher churn rates. Bundling protective services into retention offers could directly reduce attrition.

## Section 8: Payment Method Analysis

**Goal:** Examine whether the customer's payment method is associated with higher churn.

### Why payment method matters:
- Auto-pay methods (bank transfer, credit card) indicate a committed, low-friction relationship.
- Manual payment methods (electronic check, mailed check) may reflect lower engagement or intent to cancel.

In [ ]:
# ============================================================
# SECTION 8: Churn by Payment Method
# ============================================================
# PURPOSE: Compare churn rates across the 4 payment methods:
#   - Electronic check    (manual, highest churn risk)
#   - Mailed check        (manual)
#   - Bank transfer (auto)(auto-pay)
#   - Credit card (auto)  (auto-pay, lowest churn risk)
#
# bar_label() is applied to both containers (Churn=No and Churn=Yes)
# so both halves of each bar group are clearly labelled.
#
# rotation=45 on x-axis ticks prevents the long payment method labels
# from overlapping each other.
#
# BUSINESS IMPLICATION:
#   Incentivising customers to switch from electronic check to auto-pay
#   is a low-cost, high-impact retention tactic.
# ============================================================

plt.figure(figsize=(8, 5))
ax = sns.countplot(x="PaymentMethod", data=df, hue="Churn",
                   palette=['#1f77b4', '#ff7f0e'])
for container in ax.containers:
    ax.bar_label(container, fontsize=9)
plt.title("Churn by Payment Method", fontsize=13)
plt.xlabel("Payment Method")
plt.ylabel("Number of Customers")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

> **📌 Insight:** Customers paying by **electronic check** have the highest churn rate by a significant margin. Auto-pay customers (bank transfer, credit card) are considerably more loyal.

## Section 9: Key Findings Summary

| Factor | Finding | Recommended Action |
|---|---|---|
| 📋 **Contract Type** | Month-to-month customers churn most | Incentivise upgrades to annual contracts |
| ⏳ **Tenure** | Most churn happens in the first 1–6 months | Strengthen onboarding & early engagement |
| 👴 **Senior Citizens** | Proportionally higher churn among seniors | Offer dedicated senior support plans |
| 💳 **Payment Method** | Electronic check users churn most | Promote auto-pay with discounts |
| 🔒 **Services** | No OnlineSecurity / TechSupport → higher churn | Bundle protective services in retention offers |
| 🚻 **Gender** | No significant difference across genders | Gender is not an actionable churn predictor |

---

## 🚀 Next Steps

- [ ] Feature encoding & preprocessing for ML pipeline
- [ ] Predictive churn model (Logistic Regression, XGBoost, LightGBM)
- [ ] Feature importance analysis
- [ ] Customer risk scoring dashboard